<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/14-structured-streaming/04-stateful-ops.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 14.4 — Stateful ops: windows, then hand-rolled state

Tumbling vs sliding windows side by side; then `mapGroupsWithState`
tracking a running per-key count and last-seen time with an explicit
timeout -- the state you write and own, not a built-in aggregate.

In [ ]:
import os, sys, shutil, time, json
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("14.4-stateful-ops")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
src = (spark.readStream.format("rate").option("rowsPerSecond", 10).load()
       .withColumn("key", (col("value") % 4).cast("int")))

## Tumbling vs sliding: same data, different window shape

In [ ]:
tumbling = (src.withWatermark("timestamp", "5 seconds")
            .groupBy(F.window("timestamp", "4 seconds")).count())
sliding = (src.withWatermark("timestamp", "5 seconds")
           .groupBy(F.window("timestamp", "4 seconds", "2 seconds")).count())

tq = tumbling.writeStream.outputMode("update").format("memory").queryName("tumbling_demo").start()
sq = sliding.writeStream.outputMode("update").format("memory").queryName("sliding_demo").start()

time.sleep(10)
print("TUMBLING windows (non-overlapping):")
spark.sql("SELECT window.start, window.end, count FROM tumbling_demo ORDER BY window.start").show(truncate=False)
print("SLIDING windows (overlapping -- more rows, each event counted in multiple):")
spark.sql("SELECT window.start, window.end, count FROM sliding_demo ORDER BY window.start").show(truncate=False)
tq.stop(); sq.stop()

## `mapGroupsWithState` — hand-rolled running state per key

One output row per key per trigger: running count + last-seen
timestamp, persisted across micro-batches via `GroupState`.

In [ ]:
from pyspark.sql.streaming.state import GroupState, GroupStateTimeout
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, TimestampType
from collections import namedtuple

State = namedtuple("State", ["count", "last_seen"])
out_schema = StructType([
    StructField("key", IntegerType()),
    StructField("count", LongType()),
    StructField("last_seen", TimestampType()),
])
state_schema = StructType([
    StructField("count", LongType()),
    StructField("last_seen", TimestampType()),
])

def update_running_count(key, rows, state: GroupState):
    count, last_seen = (state.get.count, state.get.last_seen) if state.exists else (0, None)
    for row in rows:
        count += 1
        last_seen = row.timestamp
    state.update(State(count, last_seen))
    state.setTimeoutDuration("30 seconds")   # ProcessingTimeTimeout -- evict if silent 30s
    return (key[0], count, last_seen)

typed_result = (src.groupByKey(lambda r: (r.key,))
                .mapGroupsWithState(update_running_count,
                                     outputMode="update",
                                     timeoutConf=GroupStateTimeout.ProcessingTimeTimeout))

state_q = (typed_result.writeStream.outputMode("update")
           .format("memory").queryName("state_demo").start())

time.sleep(8)
print("running per-key state, one row per key per trigger it changed:")
spark.sql("SELECT * FROM state_demo ORDER BY key, last_seen DESC").show(20, truncate=False)
state_q.stop()

In [ ]:
spark.stop()

## Exercises

1. Change the sliding window's slide interval from 2 seconds to 1
   second (keeping the 4-second duration). How many windows does a
   single event now fall into, and how does the sliding table's row
   count change?
2. In the `mapGroupsWithState` demo, shorten the timeout to `"3
   seconds"`. Since `rate` never stops producing all 4 keys, does the
   timeout ever actually fire here? What kind of stream WOULD trigger
   it?
3. Modify `update_running_count` to track the MAX value seen per key
   instead of a running count -- still hand-rolled state, different
   aggregate logic than any built-in function provides directly for
   this shape.
4. Rewrite the running-count logic as a plain
   `groupBy("key").count()` in `update` mode. Compare the output
   shape to the `mapGroupsWithState` version -- what does the built-in
   version give you for free that you had to write by hand above?
5. Sketch (comments only, doesn't need to run) a
   `flatMapGroupsWithState` version that only emits a row once a key
   has been silent for its timeout -- i.e., a session-close event, not
   a running update every trigger.

In [ ]:
# your exercise code here